# K_01d – Reale Grid-Topologie & Netzfluss-Animationen

**Grid-Arbitrage** · Echte Übertragungs-Netzstruktur (OSM/Overpass) für CH und weitere DACH/EU-Länder

**Gruppe:** SC26_Gruppe_2 | **Status:** Kür – erweiterte Infrastruktur-Analyse | **Abhängig von:** K_01c

---

> **Ziel:** Realistische Netzknoten (Substations) und Kanten (Hochspannungsleitungen) statt synthetischer
> Zonenzentroide. Animierter Energiefluss auf echten Leitungspfaden.
>
> **Länder-generisch:** Alle Datenquellen und Funktionen sind über `CC_CODE` parametrisiert.
> Für CH ist ein vollständiger Offline-Fallback (27 Knoten, 34 Kanten, 380/220 kV) eingebaut.
>
> **Datenquellen (Priorität):**
> 1. **earth-osm** (Geofabrik PBF) — vollständigste OSM-Quelle, kein Timeout-Risiko
> 2. **Overpass API** (OSM) — direktes OSM, bei grossen Ländern ggf. Timeout
> 3. **PyPSA-Eur Zenodo** (zenodo.org/records/14144752) — preprocessed, mit elektr. Parametern
> 4. **Hardcoded Baseline** — CH-spezifisch, offline, aus Swissgrid-Publikationen
>
> ⚠️ Alle Lastfluss-Werte bleiben synthetisch modelliert.


| [← K_01c Animationen](K_01c_Energiefluss_Animationen.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_02 Cross-Border →](K_02_Cross_Border.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_01d'></a>

1. [Initialisierung](#init_K_01d)
2. [Grid-Topologie laden](#topo_K_01d)
3. [Netzwerk-Graph aufbauen](#graph_K_01d)
4. [Statische Netzwerkkarte](#static_K_01d)
5. [Fluss-Modell auf Kanten](#flow_K_01d)
6. [GIF A — Tagesfluss auf Leitungen](#gif_a_K_01d)
7. [GIF B — Jahresverlauf & Richtungsumkehr](#gif_b_K_01d)
8. [Multi-Country Erweiterung](#multi_K_01d)
9. [Abschluss](#abschluss_K_01d)


---
## 1. Initialisierung<a id='init_K_01d'></a>

[↑ TOC](#toc_K_01d)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ⚙ PFAD-KONFIGURATION — experimental/ Notebook
# Automatische Projektroot-Erkennung (experimental/ liegt parallel zu kuer/)
# ════════════════════════════════════════════════════════════════════════════
import os

EXP_BASE      = '.'
EXP_DATA_DIR  = os.path.join(EXP_BASE, 'data', 'raw')
EXP_INTER_DIR = os.path.join(EXP_BASE, 'data', 'intermediate')
EXP_CACHE_DIR = os.path.join(EXP_BASE, 'data', 'intermediate', 'grid_zenodo')
EXP_OUTPUT_DIR= os.path.join(EXP_BASE, 'output', 'charts')

def _find_project_root(start='.', max_levels=4):
    path = os.path.abspath(start)
    for _ in range(max_levels):
        if os.path.exists(os.path.join(path, 'sync', 'config.json')):
            return path
        path = os.path.dirname(path)
    return None

_root = _find_project_root()
if _root is None:
    raise RuntimeError("Projektroot nicht gefunden (sync/config.json fehlt).")

PROD_SYNC_DIR = os.path.join(_root, 'sync')
PROD_DATA_DIR = os.path.join(_root, 'data', 'raw')
PROD_INTER_DIR= os.path.join(_root, 'data', 'intermediate')

for d in [EXP_DATA_DIR, EXP_INTER_DIR, EXP_CACHE_DIR, EXP_OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

EXP_CHARTS_DIR = EXP_OUTPUT_DIR
DATA_SOURCE    = 'PyPSA-Eur Zenodo'

print(f"Projektroot: {_root}")
print(f"  PROD_DATA_DIR  = {os.path.abspath(PROD_DATA_DIR)}")
print(f"  EXP_CHARTS_DIR = {os.path.abspath(EXP_CHARTS_DIR)}")


In [ ]:
# ── Bibliotheken ─────────────────────────────────────────────────────────────
import os, warnings, json, time
import subprocess, sys
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.lines import Line2D
import requests
warnings.filterwarnings('ignore')

for pkg, imp in [('geopandas','geopandas'),('networkx','networkx'),('pillow','PIL')]:
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

import geopandas as gpd
import networkx as nx

# ── Logging-Level: INFO→WARNING damit stderr sauber bleibt ─────────────
import logging
for _log in ['earth_osm','eo','eo.stream','eo.backends',
             'matplotlib','matplotlib.animation','PIL']:
    logging.getLogger(_log).setLevel(logging.WARNING)
# matplotlib animation INFO-Meldungen zusätzlich über rcParams unterdrücken
import matplotlib as _mpl
_mpl.rcParams['animation.convert_path'] = 'convert'

print(f"GeoPandas  : {gpd.__version__}")
print(f"NetworkX   : {nx.__version__}")
print(f"📅 Ausgeführt: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


In [ ]:
# ── Config (SSOT) + Länder-Parameter ─────────────────────────────────────────
with open(os.path.join(PROD_SYNC_DIR, 'config.json')) as _f: CFG = json.load(_f)

SZ_AKTIV   = CFG['szenarien']['gleichzeitigkeit_aktiv']



# ── ⚙ Länder-Konfiguration ──────────────────────────────────────────────────
# ISO 3166-1 Alpha-2 Codes — erweiterbar auf DE, AT, FR, IT, etc.
# ⚙ config.json → kuer.k01d.laender
COUNTRY_CONFIG = {
    'CH': {
        'name':        'Schweiz',
        'osm_area':    'ISO3166-1"="CH',
        'bbox':        (5.88, 45.78, 10.60, 47.92),   # (minlon, minlat, maxlon, maxlat)
        'voltage_kv':  [220, 380],                     # zu ladende Spannungsebenen
        'zone_mapping': {                               # Kanton → Zone (aus K_01b)
            'ZH':'Nordost','TG':'Nordost','SH':'Nordost','SG':'Nordost',
            'AR':'Ostschweiz','AI':'Ostschweiz','GR':'Ostschweiz',
            'GL':'Ostschweiz','SZ':'Ostschweiz',
            'AG':'Mitte-Erzeugung','SO':'Mitte-Erzeugung','BL':'Mitte-Erzeugung',
            'BE':'Mitte-Transit','LU':'Mitte-Transit','BS':'Mitte-Transit',
            'ZG':'Mitte-Transit','NW':'Mitte-Transit','OW':'Mitte-Transit',
            'VD':'West','GE':'West','NE':'West','JU':'West','FR':'West',
            'VS':'Süd','TI':'Süd','UR':'Süd',
        },
        'zone_colors': {
            'Nordost':'#1565C0','Ostschweiz':'#00838F',
            'Mitte-Erzeugung':'#7B1FA2','Mitte-Transit':'#388E3C',
            'West':'#FF6F00','Süd':'#B71C1C',
        },
        'map_xlim':    (5.88, 10.65),
        'map_ylim':    (45.78, 47.95),
    },
    'DE': {
        'name':        'Deutschland',
        'osm_area':    'ISO3166-1"="DE',
        'bbox':        (6.00, 47.20, 15.10, 55.10),
        'voltage_kv':  [220, 380],
        'zone_mapping': {},
        'zone_colors': {},
        'map_xlim':    (6.0, 15.2),
        'map_ylim':    (47.2, 55.2),
    },
    'AT': {
        'name':        'Österreich',
        'osm_area':    'ISO3166-1"="AT',
        'bbox':        (9.50, 46.35, 17.20, 48.80),
        'voltage_kv':  [220, 380],
        'zone_mapping': {},
        'zone_colors': {},
        'map_xlim':    (9.5, 17.2),
        'map_ylim':    (46.3, 48.9),
    },
}

# ⚙ Aktives Land — hier wechseln für andere Länder
CC_CODE = 'CH'  # ⚙ config.json → kuer.k01d.aktives_land
CC = COUNTRY_CONFIG[CC_CODE]

# Farben aus config
_viz         = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK      = _viz.get('bg_dark',    '#0d1117')
BG_PANEL     = _viz.get('bg_panel',   '#141414')
C_LOAD       = _viz.get('c_load',     '#66BB6A')
C_CHARGE     = _viz.get('c_charge',   '#1565C0')
C_FEED       = _viz.get('c_feed',     '#B71C1C')
C_PRICE      = _viz.get('c_price',    '#FFA726')
C_SPINE      = _viz.get('c_spine',    '#333333')
C_ACHSE      = _viz.get('c_achse',    '#aaaaaa')
C_TICK       = _viz.get('c_tick',     '#bbbbbb')
_stil        = CFG.get('visualisierung', {}).get('stil', {})
FS_TITEL     = _stil.get('schriftgroesse_titel', 13)
FS_KLEIN     = _stil.get('schriftgroesse_klein',  7)

# Leitungsfarben nach Spannung
VOLTAGE_COLORS = {380: '#EF5350', 220: '#FFA726', 150: '#42A5F5', 110: '#66BB6A'}
VOLTAGE_LW     = {380: 2.5,       220: 1.5,       150: 1.0,       110: 0.7}

print(f"Aktives Land: {CC_CODE} — {CC['name']}")
print(f"Spannungsebenen: {CC['voltage_kv']} kV")
print(f"Cache: {CACHE_DIR}")

# Aliases für Kompatibilität
EXP_CHARTS_DIR = EXP_OUTPUT_DIR
DATA_DIR = PROD_DATA_DIR
CACHE_DIR = EXP_CACHE_DIR


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙ ANIMATIONS-SCHALTER K_01d — alle visuellen Parameter hier einstellen
# ══════════════════════════════════════════════════════════════════════════════

# ── Frame-Auflösung ───────────────────────────────────────────────────────────
_anim = CFG.get('animation', {})
FRAMES_PER_HOUR = _anim.get('frames_per_hour', 4)
FRAMES_PER_WEEK = _anim.get('frames_per_week', 4)

# ⚙ Verdopplungs-Option für flüssigere Richtungsänderungen (deaktivierbar)
FRAME_MULT      = 2          # ⚙ 1 = Config-Default | 2 = doppelte Auflösung
FRAMES_PER_HOUR = FRAMES_PER_HOUR * FRAME_MULT
FRAMES_PER_WEEK = FRAMES_PER_WEEK * FRAME_MULT

N_FRAMES_TAG    = 24 * FRAMES_PER_HOUR
N_FRAMES_JAHR   = 52 * FRAMES_PER_WEEK
FPS_TAG         = _anim.get('fps_karte', 10) * FRAME_MULT
FPS_JAHR        = _anim.get('fps_karte', 10) * FRAME_MULT
DPI_GIF         = _anim.get('dpi', 110)

# ── GIF Fluss-Dots (Kanten + Cross-Border) ────────────────────────────────────
GIF_D_NORM_MODE    = 'per_edge'  # ⚙ 'global'   = alle Kanten relativ zu GIF_D_MW_NORM_GLOBAL
                                  #    'per_edge' = jede Kante relativ zu ihrem eigenen Ø-Fluss
GIF_D_MW_NORM_GLOBAL = 800       # ⚙ Normierungswert für 'global' [MW] (typ. Kantenfluss)
GIF_D_N_DOTS_BASE  = 4           # ⚙ Anzahl Dots bei Vollauslastung
GIF_D_DOT_SIZE     = 35          # ⚙ Dot-Grösse fix [pt²]
GIF_D_MIN_MW       = 60          # ⚙ Mindestfluss für Dot-Darstellung [MW]
GIF_D_SHOW_LABEL   = False       # ⚙ MW-Label auf Kanten (sehr viele → eher aus)

print("⚙ K_01d Animations-Schalter:")
print(f"  Aufloesung: {FRAMES_PER_HOUR}f/h × 24h = {N_FRAMES_TAG}f Tag "
      f"| {FRAMES_PER_WEEK}f/W × 52W = {N_FRAMES_JAHR}f Jahr (FRAME_MULT={FRAME_MULT})")
print(f"  fps: {FPS_TAG} | norm='{GIF_D_NORM_MODE}' | "
      f"n_dots={GIF_D_N_DOTS_BASE} | dot_size={GIF_D_DOT_SIZE}")


---
## 2. Grid-Topologie laden<a id='topo_K_01d'></a>

[↑ TOC](#toc_K_01d)

Drei Quellen in Prioritätsreihenfolge: Overpass API → PyPSA-Eur Zenodo → Hardcoded Baseline.

### ⚙ Datenqualität & Granularität — auf einen Blick

| Element | Status | Quelle |
|---------|--------|--------|
| Knoten-Koordinaten | 📡 **Echt** | OSM (earth-osm/Overpass) oder Swissgrid-Baseline |
| Leitungspfade | 📡 **Echt** (wenn OSM) | OSM `power=line`, 220/380 kV |
| Lastfluss-Richtung/-Intensität | 🔬 **Modelliert** | Imbalance-Differenz, kein DC-Solver |
| Cross-Border [MW] | 🔬 **Modelliert** | Anteilsschlüssel DE/IT/FR/AT |

**Für echte Lastflüsse:** PyPSA mit Leitungsimpedanzen aus Zenodo-Dataset (enthält r, x).

#### Räumliche Granularität

| Ebene | CH | Übertragungs-Äquivalent |
|-------|----|------------------------|
| Kantone (aktuell K_01) | 26 | ~grobe Zonenstruktur |
| Bezirke | ~150 | mittelgross |
| Gemeinden | ~2'100 | feingranular, aber kein Lastfluss-Relevanz |
| **Substations (K_01d)** | **147 (Swissgrid)** | **✅ physikalisch korrekt** |

→ Substations sind die einzig sinnvolle Granularität für Netzfluss-Analyse.
Gemeinden wären für Verbrauchsverteilung interessant, nicht für Topologie.


In [ ]:
# Baseline wurde entfernt — PyPSA-Eur Zenodo ist einzige Quelle
# Bei Offline-Betrieb: zenodo.org/records/14144752 manuell herunterladen
print('Baseline nicht mehr benötigt — PyPSA-Eur Zenodo als einzige Quelle')


In [ ]:
# ── PyPSA-Eur Zenodo — einzige Topologie-Quelle ──────────────────────────────
# Quelle: zenodo.org/records/{ZENODO_RECORD}
# Enthält: Substations + Leitungen mit r/x/b/s_nom für ganz Europa (220–750 kV)
# Download: ~5 MB gesamt (ganz Europa) vs. 300 MB+ PBF pro Land
# Cache: EXP_CACHE_DIR (experimental/data/intermediate/grid_zenodo/)

import logging, requests
from io import StringIO
logging.getLogger('matplotlib').setLevel(logging.WARNING)
logging.getLogger('PIL').setLevel(logging.WARNING)

def load_pypsa_zenodo(cc_code, cache_dir, zenodo_record=ZENODO_RECORD, force=False):
    '''
    Lädt PyPSA-Eur prebuilt network von Zenodo.
    Filter: country == cc_code (z.B. 'CH', 'DE', 'AT').
    Gibt (df_nodes, df_edges) zurück.
    Alle Dateien landen in cache_dir (experimental/data/intermediate/grid_zenodo/).
    
    ⚙ Für anderes Land: cc_code anpassen (CC_CODE in Pfad-Konfiguration).
    ⚙ Neue Zenodo-Version: ZENODO_RECORD in Pfad-Konfiguration anpassen.
    '''
    buses_cache = os.path.join(cache_dir, f'{cc_code}_pypsa_buses.csv')
    lines_cache = os.path.join(cache_dir, f'{cc_code}_pypsa_lines.csv')
    raw_buses   = os.path.join(cache_dir, 'zenodo_buses_all.csv')
    raw_lines   = os.path.join(cache_dir, 'zenodo_lines_all.csv')

    # ── 1: Cache prüfen ───────────────────────────────────────────────────────
    if not force and os.path.exists(buses_cache) and os.path.exists(lines_cache):
        df_nodes = pd.read_csv(buses_cache)
        df_edges = pd.read_csv(lines_cache)
        print(f'{cc_code}: PyPSA Cache geladen ({len(df_nodes)} Knoten, {len(df_edges)} Kanten)')
        return df_nodes, df_edges

    # ── 2: Gesamtdaten downloaden (falls nicht gecacht) ───────────────────────
    base_url = f'https://zenodo.org/records/{zenodo_record}/files'

    if not force and os.path.exists(raw_buses) and os.path.exists(raw_lines):
        print(f'Zenodo Rohdaten aus Cache laden...')
        df_buses_all = pd.read_csv(raw_buses)
        df_lines_all = pd.read_csv(raw_lines)
    else:
        print(f'▶ Lade PyPSA-Eur Zenodo Record {zenodo_record} (Europa ~5 MB)...')
        rb = requests.get(f'{base_url}/buses.csv?download=1', timeout=90)
        rl = requests.get(f'{base_url}/lines.csv?download=1', timeout=90)
        if rb.status_code != 200 or rl.status_code != 200:
            print(f'⚠️  Zenodo nicht erreichbar: HTTP {rb.status_code}/{rl.status_code}')
            return None, None
        df_buses_all = pd.read_csv(StringIO(rb.text))
        df_lines_all = pd.read_csv(StringIO(rl.text))
        # Raw cachen (alle Länder) für spätere Länder-Wechsel ohne Re-Download
        df_buses_all.to_csv(raw_buses, index=False)
        df_lines_all.to_csv(raw_lines, index=False)
        print(f'  Europa gesamt: {len(df_buses_all)} Busse, {len(df_lines_all)} Leitungen')
        print(f'  Gecacht: {raw_buses}')

    # ── 3: Auf Land filtern ────────────────────────────────────────────────────
    # buses.csv Spalten: bus_id, voltage, dc, x (lon), y (lat), country, ...
    # lines.csv Spalten: line_id, bus0, bus1, x, r, b, s_nom, v_nom, ...
    country_col = 'country' if 'country' in df_buses_all.columns else None
    if country_col is None:
        # Fallback: bus_id enthält oft Länder-Prefix 'CH1-380'
        df_buses_all['country'] = df_buses_all.index.astype(str).str[:2]
        country_col = 'country'

    df_b = df_buses_all[df_buses_all[country_col] == cc_code].copy()
    if len(df_b) == 0:
        print(f'⚠️  Keine Busse für {cc_code} gefunden. Verfügbare Länder:')
        print(f'   {sorted(df_buses_all[country_col].unique())}')
        return None, None

    # Leitungen filtern: beide Endpunkte im Land
    bus_ids = set(df_b.index.astype(str).tolist() + df_b.get('bus_id', pd.Series()).astype(str).tolist())
    # lines haben bus0/bus1 als bus_id
    if 'bus0' in df_lines_all.columns:
        mask = (df_lines_all['bus0'].astype(str).isin(bus_ids) |
                df_lines_all['bus1'].astype(str).isin(bus_ids))
        df_l = df_lines_all[mask].copy()
    else:
        df_l = df_lines_all.copy()

    # Spalten normalisieren
    # buses: id, lon, lat, voltage_kv, country
    lon_col = 'x' if 'x' in df_b.columns else 'lon'
    lat_col = 'y' if 'y' in df_b.columns else 'lat'
    v_col   = 'v_nom' if 'v_nom' in df_b.columns else ('voltage' if 'voltage' in df_b.columns else None)

    df_nodes = pd.DataFrame({
        'id':         df_b.index.astype(str) if 'bus_id' not in df_b.columns else df_b['bus_id'].astype(str),
        'lon':        pd.to_numeric(df_b[lon_col], errors='coerce'),
        'lat':        pd.to_numeric(df_b[lat_col], errors='coerce'),
        'voltage_kv': pd.to_numeric(df_b[v_col], errors='coerce') if v_col else 220,
        'zone':       '',
        'name':       df_b.get('name', pd.Series([''] * len(df_b))).astype(str),
    }).dropna(subset=['lon','lat']).reset_index(drop=True)

    # lines: id, bus0, bus1, voltage_kv, r, x, b, s_nom
    df_edges = df_l[['bus0','bus1'] + 
                    [c for c in ['r','x','b','s_nom','v_nom','length'] if c in df_l.columns]
                   ].copy().reset_index(drop=True)
    df_edges['from_node'] = df_edges['bus0'].astype(str)
    df_edges['to_node']   = df_edges['bus1'].astype(str)
    if 'v_nom' in df_edges.columns:
        df_edges['voltage_kv'] = pd.to_numeric(df_edges['v_nom'], errors='coerce').fillna(220)
    else:
        df_edges['voltage_kv'] = 220
    df_edges['id'] = df_edges.index.astype(str)

    # Cachen (nur dieses Land)
    df_nodes.to_csv(buses_cache, index=False)
    df_edges.to_csv(lines_cache, index=False)

    print(f'✅ {cc_code}: {len(df_nodes)} Substations, {len(df_edges)} Leitungen')
    kv_dist = df_nodes['voltage_kv'].value_counts().head(5).to_dict()
    print(f'   Spannungen: {kv_dist}')
    return df_nodes, df_edges


# ── Kantone für Visualisierung (aus PROD, nur lesen) ─────────────────────────
def load_kantone(prod_data_dir):
    KANT_NUM_TO_ABK = {i+1:k for i,k in enumerate([
        'ZH','BE','LU','UR','SZ','OW','NW','GL','ZG','FR','SO','BS','BL',
        'SH','AR','AI','SG','GR','AG','TG','TI','VD','VS','NE','GE','JU'])}
    for path in [os.path.join(prod_data_dir,'kantone.gpkg'),
                 os.path.join(prod_data_dir,'swissboundaries3d.gpkg')]:
        if os.path.exists(path) and os.path.getsize(path)>50_000:
            try:
                import geopandas as gpd
                layers = gpd.list_layers(path)
                lname = next((l for l in layers['name'] if 'kanton' in l.lower()),
                              layers['name'].iloc[0])
                gdf = gpd.read_file(path, layer=lname)
                if gdf.crs and gdf.crs.to_epsg()!=4326: gdf=gdf.to_crs(epsg=4326)
                if 'KAB' not in gdf.columns:
                    for col in gdf.columns:
                        s = gdf[col].astype(str).str.strip()
                        if s.str.isnumeric().all():
                            gdf['KAB']=s.astype(int).map(KANT_NUM_TO_ABK); break
                        elif s.str.len().between(2,3).mean()>0.8:
                            gdf['KAB']=s.str.upper().str[:2]; break
                print(f'Kantone: {len(gdf)} | {os.path.basename(path)}')
                return gdf
            except Exception as e:
                print(f'  {e}')
    print('⚠️  Kantone nicht gefunden — K_01 zuerst ausführen')
    return None

print("✅ Zenodo-Loader bereit")


In [ ]:
# ── Topologie laden: PyPSA-Eur Zenodo ────────────────────────────────────────
# Einzige Quelle. Downloads → EXP_CACHE_DIR (experimental/data/intermediate/grid_zenodo/)
# Für anderes Land: CC_CODE in Pfad-Konfiguration ändern

FORCE_RELOAD = CFG.get('force_reload', {}).get('grid_topology', False)
print(f"▶ Lade Grid-Topologie via PyPSA-Eur Zenodo (CC_CODE={CC_CODE})...")

df_nodes, df_edges = load_pypsa_zenodo(CC_CODE, EXP_CACHE_DIR,
                                        zenodo_record=ZENODO_RECORD,
                                        force=FORCE_RELOAD)

if df_nodes is None or len(df_nodes) == 0:
    print("⚠️  Zenodo nicht verfügbar — Notebook benötigt Internetverbindung")
    print("   Alternativ: zenodo.org/records/14144752 → buses.csv/lines.csv manuell herunterladen")
    print(f"   Ziel: {EXP_CACHE_DIR}/zenodo_buses_all.csv")
else:
    DATA_SOURCE = f'PyPSA-Eur Zenodo (Record {ZENODO_RECORD})'
    print(f"\n✅ Quelle: {DATA_SOURCE}")
    print(f"   Knoten: {len(df_nodes)}")
    print(f"   Kanten: {len(df_edges)}")
    print(f"   Cache:  {EXP_CACHE_DIR}/")

# Kantone für Visualisierung (aus Produktionsdaten, nur lesen)
gdf_kant = load_kantone(PROD_DATA_DIR)


---
## 3. Netzwerk-Graph aufbauen<a id='graph_K_01d'></a>

[↑ TOC](#toc_K_01d)

NetworkX-Graph aus Substations (Knoten) und Leitungen (Kanten). Knotengrad = Anzahl Verbindungen → Proxy für lokale Netzwichtigkeit.

In [ ]:
# ── NetworkX-Graph aufbauen — mit OSM-Datenbereinigung ────────────────────────
import networkx as nx
import numpy as np

# CH Bounding Box (strict) — filtert grenznahe ausländische Nodes
BB = {'minlon': 5.88, 'maxlon': 10.65, 'minlat': 45.78, 'maxlat': 47.95}

def in_bbox(lon, lat, bb=BB, margin=0.10):
    return (bb['minlon']-margin <= lon <= bb['maxlon']+margin and
            bb['minlat']-margin <= lat <= bb['maxlat']+margin)

def clean_name(row_id, row_name):
    '''Gibt lesbaren Namen zurück: OSM-IDs werden durch Koordinaten-Ref ersetzt.'''
    n = str(row_name).strip() if row_name else ''
    if not n or n == 'nan' or n.lstrip('-').isdigit():
        return None  # wird später durch kV-Label ersetzt
    return n[:20]

# Nodes filtern + bauen
G = nx.Graph()

df_n_clean = df_nodes.copy()
# Strict bbox filter (Nodes ausserhalb CH entfernen)
mask = df_n_clean.apply(
    lambda r: in_bbox(float(r['lon']), float(r['lat'])), axis=1)
df_n_clean = df_n_clean[mask].copy()
print(f"Nodes nach bbox-Filter: {len(df_n_clean)} / {len(df_nodes)}")

# Deduplication: Nodes die < 800m auseinander liegen → zusammenführen
# (Overpass liefert oft mehrere Nodes pro physischer Substation)
from scipy.spatial import cKDTree
coords = df_n_clean[['lon','lat']].values.astype(float)
# 1 Grad ≈ 85 km in CH → 0.01° ≈ 850 m
MERGE_DEG = 0.008  # ~680 m
if len(coords) > 1:
    tree = cKDTree(coords)
    pairs = tree.query_pairs(MERGE_DEG)
    # Union-Find: Gruppen bilden
    parent = list(range(len(df_n_clean)))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b): parent[find(a)] = find(b)
    for a, b in pairs: union(a, b)
    # Repräsentant pro Gruppe: höchste Spannung + erster Eintrag
    groups = {}
    for idx in range(len(df_n_clean)):
        root = find(idx)
        groups.setdefault(root, []).append(idx)
    # Wähle Repräsentanten: max voltage, dann erste
    keep_rows = []
    for root, idxs in groups.items():
        sub = df_n_clean.iloc[idxs]
        best = sub.sort_values('voltage_kv', ascending=False).iloc[0]
        keep_rows.append(best)
    df_n_clean = pd.DataFrame(keep_rows).reset_index(drop=True)
    print(f"Nach Deduplication (≤{MERGE_DEG*111:.0f}km): {len(df_n_clean)} Nodes")

# Nodes zum Graph hinzufügen
for _, row in df_n_clean.iterrows():
    nid  = str(row['id'])
    name = clean_name(nid, row.get('name', ''))
    kv   = int(row.get('voltage_kv', 220))
    G.add_node(nid,
               lon=float(row['lon']),
               lat=float(row['lat']),
               voltage_kv=kv,
               zone=str(row.get('zone', '')),
               name=name if name else f'{kv}kV',
               display_name=name if name else None)

# Kanten: Snap Leitungs-Endpunkte auf nächste Substation ────────────────────
# Nur Kanten innerhalb CH, maximale Snap-Distanz 25 km
MAX_SNAP_DEG = 0.25  # ~21 km
node_ids   = list(G.nodes())
node_lons  = np.array([G.nodes[n]['lon'] for n in node_ids])
node_lats  = np.array([G.nodes[n]['lat'] for n in node_ids])
node_tree  = cKDTree(np.column_stack([node_lons, node_lats]))

added_edges = set()
skipped = 0

for _, row in df_edges.iterrows():
    try:
        fl, flo = float(row['from_lat']), float(row['from_lon'])
        tl, tlo = float(row['to_lat']),  float(row['to_lon'])
    except (KeyError, ValueError):
        skipped += 1; continue

    # Beide Endpunkte müssen in CH bbox liegen
    if not in_bbox(flo, fl) or not in_bbox(tlo, tl):
        skipped += 1; continue

    # Snap auf nächste Substations
    fd, fi = node_tree.query([flo, fl])
    td, ti = node_tree.query([tlo, tl])

    if fd > MAX_SNAP_DEG or td > MAX_SNAP_DEG:
        skipped += 1; continue

    n1, n2 = node_ids[fi], node_ids[ti]
    if n1 == n2: continue

    key = (min(n1,n2), max(n1,n2))
    kv  = int(row.get('voltage_kv', 220))
    if key not in added_edges:
        G.add_edge(n1, n2, voltage_kv=kv)
        added_edges.add(key)

print(f"Kanten: {G.number_of_edges()} hinzugefügt, {skipped} übersprungen")

# Isolierte Nodes entfernen (Overpass liefert oft Single-Substations ohne Leitungen)
isolated = list(nx.isolates(G))
G.remove_nodes_from(isolated)
print(f"Isolierte Nodes entfernt: {len(isolated)}")

# Knotengrad
degree = dict(G.degree())
nx.set_node_attributes(G, degree, 'degree')

major    = [n for n,d in degree.items() if d >= 3]
minor    = [n for n,d in degree.items() if d == 2]
isolated_r = [n for n,d in degree.items() if d <= 1]

print(f"\nGraph final: {G.number_of_nodes()} Knoten, {G.number_of_edges()} Kanten")
print(f"  Grosse Knoten (Grad ≥3): {len(major)}")
print(f"  Mittlere Knoten (Grad 2): {len(minor)}")
if major:
    top5 = sorted(degree.items(), key=lambda x: -x[1])[:5]
    print(f"  Top-5 nach Grad: {top5}")


---
## 4. Statische Netzwerkkarte<a id='static_K_01d'></a>

[↑ TOC](#toc_K_01d)

Leitungen nach Spannung (Farbe + Stärke), Knoten nach Grad (Grösse).

In [ ]:
# Kantone bereits in load-topo Zelle geladen (gdf_kant)
# Falls gdf_kant None: K_01 zuerst ausführen
if gdf_kant is None:
    gdf_kant = load_kantone(PROD_DATA_DIR)


In [ ]:
print("\n▶ Starte Erstellung: Statische Netzwerkkarte (1/3)")
# 📡 ECHTE TOPOLOGIE (earth-osm/Overpass/Zenodo/Baseline) | 🔬 SYNTHETISCHE Zonenfarben
# ── Statische Netzwerkkarte ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor('#090d14')
ax.set_xlim(*CC['map_xlim']); ax.set_ylim(*CC['map_ylim'])
ax.set_axis_off()

# Kantonsgrenzen
if gdf_kant is not None:
    gdf_kant.boundary.plot(ax=ax, color='#1e2d40', linewidth=0.7, zorder=1)
    try:
        gpd.GeoDataFrame(geometry=[gdf_kant.unary_union.boundary], crs='EPSG:4326').plot(
            ax=ax, color='#2a3d55', linewidth=2.0, zorder=2)
    except Exception:
        pass

# Kanten nach Spannung (dicker = höher)
for n1, n2, data in G.edges(data=True):
    kv  = data.get('voltage_kv', 220)
    col = VOLTAGE_COLORS.get(kv, '#888')
    lw  = VOLTAGE_LW.get(kv, 1.0)
    x1, y1 = G.nodes[n1]['lon'], G.nodes[n1]['lat']
    x2, y2 = G.nodes[n2]['lon'], G.nodes[n2]['lat']
    ax.plot([x1,x2],[y1,y2], color=col, lw=lw, alpha=0.75, zorder=3, solid_capstyle='round')

# Knoten: Grösse = Grad, Farbe = Zone oder Spannung
zone_col = CC.get('zone_colors', {})
for n, data in G.nodes(data=True):
    deg  = data.get('degree', 1)
    kv   = data.get('voltage_kv', 220)
    zone = data.get('zone', '')
    col  = zone_col.get(zone, VOLTAGE_COLORS.get(kv, '#ccc'))
    size = max(25, min(400, deg**2 * 15))
    ax.scatter(data['lon'], data['lat'], s=size, c=col,
               alpha=0.92, zorder=10, linewidths=0.5,
               edgecolors='white' if deg >= 3 else 'none')
    # Labels für grosse Knoten
    if deg >= 3:
        name = data.get('display_name') or data.get('name', '')
        if not name: continue
        name = name[:14]
        ax.text(data['lon']+0.04, data['lat']+0.03, name,
                color='white', fontsize=5.5, zorder=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', fc='#090d14', alpha=0.7, lw=0))

# Legende
volt_handles = [Line2D([0],[0], color=VOLTAGE_COLORS[kv], lw=VOLTAGE_LW[kv]+0.5,
                       label=f'{kv} kV') for kv in [380,220]]
size_handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor='#aaa',
                       markersize=ms, label=f'Grad {d}')
                for ms, d in [(4,1),(7,3),(10,5)]]
ax.legend(handles=volt_handles+size_handles, loc='lower left',
          fontsize=7, framealpha=0.6, facecolor='#111', labelcolor='white')

ax.set_title(f'Übertragungnetz {CC["name"]} — {CC_CODE} | Quelle: {DATA_SOURCE}',
             color='white', fontsize=FS_TITEL, fontweight='bold')
fig.text(0.98, 0.02, '⚠️ Topologie: OSM/Baseline | Lastfluss: synthetisch',
         ha='right', va='bottom', color='#ff5252', fontsize=7, transform=fig.transFigure)

plt.tight_layout()
p = os.path.join(EXP_CHARTS_DIR, f'EXP_kuer_k01d_{CC_CODE.lower()}_netz_statisch.png')
plt.savefig(p, dpi=120, bbox_inches='tight', facecolor=BG_DARK)
print(f"✅ Gespeichert: {p}")


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
from IPython.display import Image, display
import os
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01d_ch_netz_statisch.png')
if os.path.exists(_p):
    display(Image(filename=_p))
else:
    print(f'⚠️  Datei nicht gefunden: {_p}')


**Statische Netzwerkkarte CH**

Übertragungnetz CH. Leitungen: 380 kV (rot/dick), 220 kV (orange/dünn).
Knoten-Grösse = Netzgrad (wichtige Hubs: Beznau, Mettlen, Laufenburg).
Quelle: Echte OSM-Topologie wenn online, Swissgrid-Baseline (27 Knoten) wenn offline.

---
## 5. Fluss-Modell auf Kanten<a id='flow_K_01d'></a>

[↑ TOC](#toc_K_01d)

Synthetisches Lastfluss-Modell: Imbalance pro Zone → Fluss auf kürzestem Pfad im Graph.
Richtung und Intensität variieren saisonal und stündlich — **Richtungsumkehr ist modelliert**.

In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline

# Zeitachsen für Animationen
TAG_TIMES  = np.linspace(0, 24, N_FRAMES_TAG,  endpoint=False)
JAHR_TIMES = np.linspace(0, 52, N_FRAMES_JAHR, endpoint=False)
MONAT_KURZ = ['Jan','Feb','Mrz','Apr','Mai','Jun','Jul','Aug','Sep','Okt','Nov','Dez']

# ── Zonen-Imbalance Zeitreihen (aus K_01c-Logik) ─────────────────────────────
HOURS = np.arange(24)
WEEKS = np.arange(52)
SAISONS = ['Winter','Frühling','Sommer','Herbst']
MONAT_KURZ = ['Jan','Feb','Mrz','Apr','Mai','Jun','Jul','Aug','Sep','Okt','Nov','Dez']

# Installierte Kapazitäten [MW] und Verbrauch aus K_01c-Modell
ZONE_PROD_INSTALLED = {
    'Nordost':         {'Solar':1800,'Wasserkraft': 500,'Kernkraft':   0},
    'Ostschweiz':      {'Solar': 600,'Wasserkraft':2800,'Kernkraft':   0},
    'Mitte-Erzeugung': {'Solar':1200,'Wasserkraft': 800,'Kernkraft':5000},
    'Mitte-Transit':   {'Solar':1400,'Wasserkraft':1200,'Kernkraft':   0},
    'West':            {'Solar':1000,'Wasserkraft': 900,'Kernkraft':   0},
    'Süd':             {'Solar': 800,'Wasserkraft':8500,'Kernkraft':   0},
}
CF = {'Solar':0.12,'Wasserkraft':0.38,'Kernkraft':0.80}
CF_SEASONAL = {
    'Solar':      {'Winter':0.30,'Frühling':0.75,'Sommer':1.00,'Herbst':0.50},
    'Wasserkraft':{'Winter':0.65,'Frühling':0.95,'Sommer':0.90,'Herbst':0.72},
    'Kernkraft':  {'Winter':1.00,'Frühling':0.98,'Sommer':0.92,'Herbst':0.98},
}
KANTON_POP = {
    'ZH':1593583,'BE':1065607,'LU':433654,'UR':37208,'SZ':165539,'OW':39028,
    'NW':43921,'GL':40590,'ZG':130426,'FR':340765,'SO':279375,'BS':183709,
    'BL':297025,'SH':86928,'AR':58050,'AI':16293,'SG':530468,'GR':202461,
    'AG':718084,'TG':295373,'TI':358353,'VD':826400,'VS':357043,'NE':177589,
    'GE':511655,'JU':73584,
}
KANTON_TO_ZONE_B = CC['zone_mapping']
zone_pop = {}
for k, z in KANTON_TO_ZONE_B.items():
    zone_pop[z] = zone_pop.get(z,0) + KANTON_POP.get(k,0)
SPEZ_KW = 0.76

ZONE_BASE_CONS = {z: pop * SPEZ_KW / 1000 for z,pop in zone_pop.items()}

def solar_h(h): return max(0.0, float(np.sin(np.pi*(h-6.0)/13.0)))
def hydro_h(h): return float(1.0 + 0.22*np.exp(-((h-10.5)**2)/14) + 0.18*np.exp(-((h-19.0)**2)/10))
def cons_h(h, sais):
    s = {'Winter':1.15,'Frühling':1.00,'Sommer':0.87,'Herbst':1.02}.get(sais,1.0)
    return s*(1.0 + 0.38*np.exp(-((h-8.5)**2)/4.5) + 0.44*np.exp(-((h-19.0)**2)/5.0) - 0.28*np.exp(-((h-4.0)**2)/3.0))

def solar_w(w): return 0.12 + 0.10*np.sin(2*np.pi*(w-12)/52)
def hydro_w(w): return 0.38 + 0.14*np.sin(2*np.pi*(w-15)/52)
def nuclear_w(w): return 0.79 - 0.05*np.sin(2*np.pi*(w-26)/52)
def cons_w(w): return 1.00 - 0.14*np.sin(2*np.pi*(w-12)/52)

def zone_imb_h(zone, h, sais):
    prod = sum(inst * CF[et] * CF_SEASONAL[et][sais]
               * (solar_h(h) if et=='Solar' else (hydro_h(h) if et=='Wasserkraft' else 1.0))
               for et, inst in ZONE_PROD_INSTALLED.get(zone,{}).items())
    cons = ZONE_BASE_CONS.get(zone, 500) * cons_h(h, sais)
    return prod - cons

def zone_imb_w(zone, w):
    prod = sum(inst * (solar_w(w) if et=='Solar' else (hydro_w(w) if et=='Wasserkraft' else nuclear_w(w)))
               for et, inst in ZONE_PROD_INSTALLED.get(zone,{}).items())
    cons = ZONE_BASE_CONS.get(zone, 500) * cons_w(w)
    return prod - cons

# Kantonsgrenze CH: Winter Nacht-Stunde importiert CH?
print("Imbalance-Test Nordost (= Verbrauchszentrum):")
for sais in SAISONS:
    imb_00 = zone_imb_h('Nordost', 0, sais)
    imb_12 = zone_imb_h('Nordost', 12, sais)
    print(f"  {sais}: 00h={imb_00:+.0f} MW, 12h={imb_12:+.0f} MW")

print("\nCH Gesamt-Imbalance (Export+):")
for sais in SAISONS:
    total = sum(zone_imb_h(z, 0, sais) for z in ZONE_PROD_INSTALLED)
    print(f"  {sais} 00h: CH gesamt {total:+.0f} MW")

print("\nJahresverlauf CH gesamt (Winter↔Sommer):")
for w in [0, 13, 26, 39]:
    total_w = sum(zone_imb_w(z, w) for z in ZONE_PROD_INSTALLED)
    mn = MONAT_KURZ[int(w/52*12)]
    print(f"  Woche {w+1:02d} ({mn}): {total_w:+.0f} MW")


In [ ]:
# ── Kantens-Fluss auf Graph (physikalisch vereinfacht) ────────────────────────
# Modell: Überschuss-Zone exportiert via kürzestem Graphpfad zur Defizit-Zone.
# Mehrere Pfade überlagern sich → Kanten-Fluss = Summe der Teilflüsse.
# Richtung kann wechseln (Sommer: CH exportiert nach IT, Winter: importiert aus DE)

def compute_edge_flows(G, zone_fn, cc_cfg):
    '''
    Berechnet synthetischen Fluss auf jeder Kante.
    zone_fn(zone_name) -> imbalance_mw
    Gibt dict {(n1,n2): mw_flow} zurück (positiv = n1→n2).
    '''
    # Zonen-Imbalance
    zone_imb = {}
    for zone in cc_cfg.get('zone_colors', {}).keys():
        zone_imb[zone] = zone_fn(zone)

    # Knoten-Imbalance: Zone-Imbalance gleichmässig auf Knoten der Zone verteilt
    node_zone = nx.get_node_attributes(G, 'zone')
    zone_node_count = {}
    for n, z in node_zone.items():
        zone_node_count[z] = zone_node_count.get(z, 0) + 1

    node_surplus = {}
    for n, z in node_zone.items():
        cnt = zone_node_count.get(z, 1)
        node_surplus[n] = zone_imb.get(z, 0) / max(cnt, 1)

    # Für jeden Paarung (Überschuss→Defizit): kürzester Pfad, Fluss anteilig
    edge_flow = {(min(u,v),max(u,v)): 0.0 for u,v in G.edges()}

    surplus_nodes = [(n, s) for n,s in node_surplus.items() if s > 50]
    deficit_nodes = [(n, s) for n,s in node_surplus.items() if s < -50]

    for s_node, s_mw in surplus_nodes:
        for d_node, d_mw in deficit_nodes:
            if s_node == d_node: continue
            try:
                path = nx.shortest_path(G, s_node, d_node)
            except nx.NetworkXNoPath:
                continue
            # Fluss-Anteil proportional zum kleineren der beiden Werte
            flow = min(abs(s_mw), abs(d_mw)) * 0.3  # Dämpfung
            for i in range(len(path)-1):
                u, v = path[i], path[i+1]
                key = (min(u,v), max(u,v))
                # Richtung: positive = u→v
                if u < v:
                    edge_flow[key] = edge_flow.get(key, 0) + flow
                else:
                    edge_flow[key] = edge_flow.get(key, 0) - flow

    return edge_flow


# Cross-Border-Flüsse (vereinfacht wie K_01c)
BORDER_POINTS = {'DE':(7.60,47.72),'AT':(9.55,47.48),'IT':(8.95,45.92),'FR':(6.10,46.22)}
BORDER_COLORS = {'DE':'#42A5F5','AT':'#EF5350','IT':'#66BB6A','FR':'#FFA726'}
BORDER_WEIGHT  = {'DE':0.35,'AT':0.15,'IT':0.30,'FR':0.20}
CH_BORDER_NODES = {'DE':'Laufenburg','AT':'Pradella','IT':'Castione','FR':'Romanel'}

def get_border_flows(total_imb_mw):
    return {nb: total_imb_mw * w for nb, w in BORDER_WEIGHT.items()}

# Test
print("Edge-Flow Test Winter 00h:")
ef = compute_edge_flows(G,
     lambda z: zone_imb_h(z, 0, 'Winter'), CC)
top_flows = sorted(ef.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
for edge, flow in top_flows:
    print(f"  {edge[0]} → {edge[1]}: {flow:+.0f} MW")


In [ ]:
# ── Vorberechnete Kanten-Normierung (per_edge Modus) ─────────────────────────
# Berechnet Ø |flow| je Kante über alle Saisons + Stunden → stabile Normierung
# Einmalig vor GIF-Erstellung, dann als EDGE_NORMS dict verfügbar
import numpy as _np_en

print("Berechne EDGE_NORMS (per_edge Normierung)...", end=' ')
_sample_hours = _np_en.linspace(0, 24, 12, endpoint=False)
_edge_flow_acc = {k: [] for k in compute_edge_flows(
    G, lambda z: zone_imb_h(z, 12, 'Sommer'), CC).keys()}

for _saison in SAISONS:
    for _h in _sample_hours:
        _ef = compute_edge_flows(G, lambda z: zone_imb_h(z, _h, _saison), CC)
        for k, v in _ef.items():
            if k in _edge_flow_acc:
                _edge_flow_acc[k].append(abs(v))

EDGE_NORMS = {}
for k, vals in _edge_flow_acc.items():
    EDGE_NORMS[k] = max(float(_np_en.mean(vals)) if vals else 0, GIF_D_MIN_MW * 2)

# Globalwert für Border
_all_norms = [v for v in EDGE_NORMS.values() if v > 0]
BORDER_NORM_D = max(float(_np_en.mean(_all_norms)) if _all_norms else GIF_D_MW_NORM_GLOBAL, 100)

n_active = sum(1 for v in EDGE_NORMS.values() if v > GIF_D_MIN_MW)
print(f"OK — {len(EDGE_NORMS)} Kanten | {n_active} aktiv | Ø={BORDER_NORM_D:.0f} MW")


---
## 6. GIF A — Tagesfluss auf Leitungen<a id='gif_a_K_01d'></a>

[↑ TOC](#toc_K_01d)

Moving Dots auf echten Leitungspfaden. Dot-Dichte & Grösse = MW-Fluss. 4 GIFs (je Jahreszeit) × 24 Frames.

In [ ]:
# ── Karte + Netz + Schnelles Rendering (gecachter Hintergrund) ───────────────
import io, numpy as np
from PIL import Image as PILImage
import matplotlib
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

MAP_XLIM = CC['map_xlim']
MAP_YLIM = CC['map_ylim']

# Zeitachsen: N_FRAMES Punkte gleichmässig über 24h / 52 Wochen
TAG_TIMES  = np.linspace(0, 24, N_FRAMES_TAG,  endpoint=False)
JAHR_TIMES = np.linspace(0, 52, N_FRAMES_JAHR, endpoint=False)
MONAT_KURZ = ['Jan','Feb','Mrz','Apr','Mai','Jun','Jul','Aug','Sep','Okt','Nov','Dez']

# ── Hintergrund-Cache ─────────────────────────────────────────────────────────
_BG_CACHE_D = {}

def _render_grid_background():
    '''Rendert statische Karte + Leitungen + Knoten einmalig als PIL-Image.'''
    if 'grid' in _BG_CACHE_D:
        return _BG_CACHE_D['grid']
    print("  Pre-render Grid-Hintergrund...", end=' ')

    fig = plt.figure(figsize=(14, 8), facecolor='#0d1117')
    ax  = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(*MAP_XLIM); ax.set_ylim(*MAP_YLIM)
    ax.set_facecolor('#090d14'); ax.set_axis_off()

    # Kantone
    if gdf_kant is not None:
        gdf_kant.boundary.plot(ax=ax, color='#1a2535', linewidth=0.5, zorder=1)

    # Leitungen (statisch, ohne Fluss-Intensität)
    for n1, n2, data in G.edges(data=True):
        kv  = data.get('voltage_kv', 220)
        x1, y1 = G.nodes[n1]['lon'], G.nodes[n1]['lat']
        x2, y2 = G.nodes[n2]['lon'], G.nodes[n2]['lat']
        if (MAP_XLIM[0]-0.05<=x1<=MAP_XLIM[1]+0.05 and
            MAP_XLIM[0]-0.05<=x2<=MAP_XLIM[1]+0.05 and
            MAP_YLIM[0]-0.05<=y1<=MAP_YLIM[1]+0.05 and
            MAP_YLIM[0]-0.05<=y2<=MAP_YLIM[1]+0.05):
            ax.plot([x1,x2],[y1,y2],
                    color=VOLTAGE_COLORS.get(kv,'#888'),
                    lw=VOLTAGE_LW.get(kv,1)*0.7, alpha=0.30, zorder=2)

    # Knoten (statisch)
    zone_col = CC.get('zone_colors', {})
    for n, data in G.nodes(data=True):
        deg  = data.get('degree', 1)
        zone = data.get('zone', '')
        kv   = data.get('voltage_kv', 220)
        col  = zone_col.get(zone, VOLTAGE_COLORS.get(kv, '#ccc'))
        size = max(12, min(180, deg**2 * 10))
        ax.scatter(data['lon'], data['lat'], s=size, c=col, alpha=0.45,
                   zorder=8, linewidths=0.4 if deg>=3 else 0,
                   edgecolors='white' if deg>=3 else 'none')
        if deg >= 3 and data.get('display_name'):
            ax.text(data['lon']+0.04, data['lat']+0.03,
                    data['display_name'][:14],
                    color='#aaaaaa', fontsize=4.5, zorder=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.08', fc='#090d14', alpha=0.6, lw=0))

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=DPI_GIF, facecolor='#0d1117')
    plt.close(fig)
    buf.seek(0)
    img = PILImage.open(buf).convert('RGBA').copy()
    _BG_CACHE_D['grid'] = img
    print(f"OK ({img.size[0]}×{img.size[1]}px)")
    return img


def make_gif_fast_d(draw_dynamic_fn, n_frames, fps, path):
    '''
    Schneller GIF-Builder: gecachte statische Karte + dynamisches Overlay.
    Einzelframes → <gif_name>_frames/ Unterordner.
    draw_dynamic_fn(ax, frame_idx, n_frames)
    '''
    bg = _render_grid_background()
    frame_dir = path.replace('.gif', '_frames')
    os.makedirs(frame_dir, exist_ok=True)

    frames_pil = []
    for fi in range(n_frames):
        fig = plt.figure(figsize=(14, 8), facecolor=(0,0,0,0))
        ax  = fig.add_axes([0, 0, 1, 1])
        ax.set_xlim(*MAP_XLIM); ax.set_ylim(*MAP_YLIM)
        ax.set_facecolor((0,0,0,0)); ax.patch.set_alpha(0); ax.set_axis_off()

        draw_dynamic_fn(ax, fi, n_frames)

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=DPI_GIF,
                    facecolor=(0,0,0,0), transparent=True,
                    bbox_inches=None, pad_inches=0)
        plt.close(fig)
        buf.seek(0)
        overlay = PILImage.open(buf).convert('RGBA').copy()
        if bg.size != overlay.size:
            overlay = overlay.resize(bg.size, PILImage.LANCZOS)

        result = bg.copy(); result.paste(overlay, mask=overlay.split()[3])
        frame  = result.convert('RGB')
        frame.save(os.path.join(frame_dir, f'frame_{fi:04d}.png'), optimize=True)
        frames_pil.append(frame)

    frames_pil[0].save(path, save_all=True, append_images=frames_pil[1:],
                       duration=int(1000/fps), loop=0, optimize=True)
    kb = os.path.getsize(path)//1024
    print(f"✅ {os.path.basename(path)}")
    print(f"   {n_frames}f @{fps}fps = {n_frames/fps:.1f}s | {kb} KB")
    print(f"   Frames: {frame_dir}/")


# ── Fluss-Dots auf Netzwerk-Kanten ───────────────────────────────────────────
def draw_flow_dots_on_edge(ax, n1, n2, mw_flow, frame_idx, n_frames, min_mw=None):
    """
    Phase-Velocity Dots auf Netzwerkkante n1→n2.
    Geschwindigkeit + Anzahl proportional zu mw_flow / Normierung.
    Richtungswechsel fliessend (Phase kontinuierlich, kein Teleport).
    GIF_D_NORM_MODE steuert ob global oder per_edge normiert wird.
    """
    if min_mw is None: min_mw = GIF_D_MIN_MW
    if abs(mw_flow) < min_mw: return
    x1,y1 = G.nodes[n1]['lon'], G.nodes[n1]['lat']
    x2,y2 = G.nodes[n2]['lon'], G.nodes[n2]['lat']
    kv    = G.edges[n1,n2].get('voltage_kv', 220) if G.has_edge(n1,n2) else 220
    col   = VOLTAGE_COLORS.get(kv, '#FFA726')

    # Normierung
    edge_key = (min(n1,n2), max(n1,n2))
    if GIF_D_NORM_MODE == 'global':
        mw_norm = GIF_D_MW_NORM_GLOBAL
    else:
        mw_norm = EDGE_NORMS.get(edge_key, GIF_D_MW_NORM_GLOBAL)

    phase_vel = mw_flow / mw_norm         # signed → Richtung
    rel_flow  = abs(mw_flow) / mw_norm
    n_dots    = max(1, min(6, int(rel_flow * GIF_D_N_DOTS_BASE) + 1))

    # p1→p2 ist kanonische Richtung (n1→n2)
    p1 = np.array([x1, y1]); p2 = np.array([x2, y2])
    phase = frame_idx * phase_vel / n_frames
    for i in range(n_dots):
        t   = (phase + i/n_dots) % 1.0
        pos = p1 + t*(p2-p1)
        ax.scatter(*pos, s=GIF_D_DOT_SIZE, c=col,
                   alpha=0.90, zorder=12, linewidths=0, rasterized=True)

    if GIF_D_SHOW_LABEL and abs(mw_flow) > mw_norm * 0.3:
        mid = (p1+p2)/2
        ax.text(mid[0], mid[1]+0.03, f'{abs(mw_flow):.0f}',
                ha='center', va='bottom', color=col, fontsize=4, zorder=14,
                bbox=dict(boxstyle='round,pad=0.04', fc='#090d14', alpha=0.6, lw=0))


def draw_border_flow_d(ax, total_imb, frame_idx, n_frames):
    """Phase-Velocity Dots Cross-Border (gleiche Logik wie Kanten)."""
    bflows = {nb_key: total_imb * w for nb_key, w in BORDER_WEIGHT.items()}
    for nb_key, mw in bflows.items():
        if abs(mw) < GIF_D_MIN_MW: continue
        bl, ba = BORDER_POINTS[nb_key]
        anchor = CH_BORDER_NODES.get(nb_key)
        ax_lon, ax_lat = (G.nodes[anchor]['lon'], G.nodes[anchor]['lat'])                          if anchor and anchor in G else (8.15, 46.90)
        col = BORDER_COLORS[nb_key]

        mw_norm = BORDER_NORM_D if GIF_D_NORM_MODE == 'per_edge' else GIF_D_MW_NORM_GLOBAL
        phase_vel = mw / mw_norm
        rel_flow  = abs(mw) / mw_norm
        n_dots    = max(1, min(4, int(rel_flow * GIF_D_N_DOTS_BASE)))

        p1 = np.array([ax_lon, ax_lat]); p2 = np.array([bl, ba])
        if mw < 0: p1, p2 = p2, p1
        phase = frame_idx * (mw/mw_norm) / n_frames
        for i in range(n_dots):
            t   = (phase + i/n_dots) % 1.0
            pos = p1 + t*(p2-p1)
            ax.scatter(*pos, s=GIF_D_DOT_SIZE*0.7, c=col,
                       alpha=0.88, zorder=11, linewidths=0, rasterized=True)

        ax.plot([ax_lon,bl],[ax_lat,ba], color=col, lw=0.5, alpha=0.18, zorder=4)
        ax.text(bl, ba+0.10, f"{nb_key}" + "\n" + f"{mw:+.0f}MW",
                ha='center', va='bottom', color=col, fontsize=5, fontweight='bold',
                zorder=15, bbox=dict(boxstyle='round,pad=0.1', fc='#090d14', alpha=0.75, lw=0))


def add_timestamp_bar_d(ax, label_left, label_center, label_right):
    xspan = MAP_XLIM[1]-MAP_XLIM[0]; yspan = MAP_YLIM[1]-MAP_YLIM[0]
    ax.text(MAP_XLIM[0]+0.01*xspan, MAP_YLIM[1]-0.02*yspan,
            label_left, ha='left', va='top', color='#aaa', fontsize=8, zorder=20)
    ax.text(MAP_XLIM[0]+0.5*xspan,  MAP_YLIM[1]-0.02*yspan,
            label_center, ha='center', va='top', color='white', fontsize=10,
            fontweight='bold', zorder=20)
    ax.text(MAP_XLIM[1]-0.01*xspan, MAP_YLIM[1]-0.02*yspan,
            label_right, ha='right', va='top', color='#ff5252', fontsize=7, zorder=20)

print(f"✅ K_01d Helper + Fast-Engine geladen")
print(f"   Tag:  {N_FRAMES_TAG}f ({FRAMES_PER_HOUR}f/h × 24h) @{FPS_TAG}fps = {N_FRAMES_TAG/FPS_TAG:.1f}s")
print(f"   Jahr: {N_FRAMES_JAHR}f ({FRAMES_PER_WEEK}f/Woche × 52W) @{FPS_JAHR}fps = {N_FRAMES_JAHR/FPS_JAHR:.1f}s")


In [ ]:
print("\n▶ Starte Erstellung: Tagesfluss 4 Saisons (2/3)")
# 📡 ECHTE TOPOLOGIE | 🔬 SYNTHETISCHER Lastfluss | Cubic-interpoliert

def _dyn_gif_a(ax, fi, nf, saison):
    t_h    = TAG_TIMES[fi]; h_disp = int(round(t_h)) % 24
    ef     = compute_edge_flows(G, lambda z: zone_imb_h(z, t_h, saison), CC)
    total  = sum(zone_imb_h(z, t_h, saison) for z in ZONE_PROD_INSTALLED)

    # Kanten-Fluss-Dots
    for (n1, n2), mw in ef.items():
        draw_flow_dots_on_edge(ax, n1, n2, mw, fi, nf)

    # Cross-Border
    draw_border_flow_d(ax, total, fi, nf)

    # Zone-Imbalance Labels
    zone_col = CC.get('zone_colors', {})
    for zone, col in zone_col.items():
        z_nodes = [n for n,d in G.nodes(data=True) if d.get('zone')==zone]
        if z_nodes:
            cx = np.mean([G.nodes[n]['lon'] for n in z_nodes])
            cy = np.mean([G.nodes[n]['lat'] for n in z_nodes])
            imb = zone_imb_h(zone, t_h, saison)
            ax.text(cx, cy, f'{imb:+.0f}', ha='center', va='center',
                    color='white', fontsize=5.5, fontweight='bold', zorder=14,
                    bbox=dict(boxstyle='round,pad=0.12', fc=col, alpha=0.65, lw=0))

    # CH-Saldo
    col_s = '#66BB6A' if total > 0 else '#EF5350'
    ax.text((MAP_XLIM[0]+MAP_XLIM[1])/2, MAP_YLIM[1]-0.25,
            f'CH: {total:+.0f} MW', ha='center', va='top',
            color='white', fontsize=7, fontweight='bold', zorder=16,
            bbox=dict(boxstyle='round,pad=0.2', fc=col_s, alpha=0.80, lw=0))

    add_timestamp_bar_d(ax, f"⚙ Modelliert | Quelle: {DATA_SOURCE}",
                        f"Lastfluss {CC['name']} — {h_disp:02d}:00 ({saison})",
                        "⚠️ Synthetische Lastflüsse")

for saison in SAISONS:
    p = os.path.join(EXP_CHARTS_DIR, f'EXP_kuer_k01d_{CC_CODE.lower()}_tag_{saison.lower()}.gif')
    make_gif_fast_d(lambda ax,fi,nf,s=saison: _dyn_gif_a(ax,fi,nf,s),
                    N_FRAMES_TAG, FPS_TAG, p)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
from IPython.display import Image, display
import os
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01d_ch_tag_winter.gif')
if os.path.exists(_p):
    display(Image(filename=_p))
else:
    print(f'⚠️  Datei nicht gefunden: {_p}')


**GIF A — Tagesfluss auf echten Leitungspfaden (Winter)**

Moving Dots auf echten OSM-Leitungen. Dot-Dichte = MW-Fluss.
Zone-Labels ±MW. Cross-Border an echten Grenzknoten (Laufenburg→DE, Castione→IT).

---
## 7. GIF B — Jahresverlauf & Richtungsumkehr<a id='gif_b_K_01d'></a>

[↑ TOC](#toc_K_01d)

52 Wochen. Die CH-Bilanz kippt von Export (Sommer) zu Import (Winter-Spitze). Richtungsumkehr der Cross-Border-Dots sichtbar.

In [ ]:
print("\n▶ Starte Erstellung: Jahresverlauf 19h + 12h (3/3)")
# 📡 ECHTE TOPOLOGIE | 🔬 SYNTHETISCH | ✅ RICHTUNGSUMKEHR modelliert

def _dyn_gif_b(ax, fi, nf, hour_of_day):
    t_w    = JAHR_TIMES[fi]; monat = MONAT_KURZ[int(t_w/52*12)%12]
    saison_idx = int((t_w/52)*4); saison = SAISONS[saison_idx%4]
    week_scale = float(cons_w(t_w))

    ef    = compute_edge_flows(G, lambda z: zone_imb_h(z, hour_of_day, saison)*week_scale, CC)
    total = sum(zone_imb_h(z, hour_of_day, saison)*week_scale for z in ZONE_PROD_INSTALLED)

    for (n1, n2), mw in ef.items():
        draw_flow_dots_on_edge(ax, n1, n2, mw, fi, nf)
    draw_border_flow_d(ax, total, fi, nf)

    # CH-Saldo mit Richtungsindikator
    col_s = '#66BB6A' if total > 0 else '#EF5350'
    richtung = 'Export ↗' if total > 0 else 'Import ↙'
    ax.text((MAP_XLIM[0]+MAP_XLIM[1])/2, MAP_YLIM[1]-0.25,
            f'CH: {total:+.0f} MW\n{richtung}', ha='center', va='top',
            color='white', fontsize=7, fontweight='bold', zorder=16,
            bbox=dict(boxstyle='round,pad=0.2', fc=col_s, alpha=0.80, lw=0))

    add_timestamp_bar_d(ax, f"⚙ Modelliert | {DATA_SOURCE}",
                        f"Jahresverlauf {CC['name']} — Woche {int(t_w)+1:02d}/52 ({monat}) | {hour_of_day:02d}:00",
                        "⚠️ Synthetische Lastflüsse")

# 19h — Abendspitze (Winter: Importrisiko sichtbar)
p19 = os.path.join(EXP_CHARTS_DIR, f'EXP_kuer_k01d_{CC_CODE.lower()}_jahr_19h.gif')
make_gif_fast_d(lambda ax,fi,nf: _dyn_gif_b(ax,fi,nf,19), N_FRAMES_JAHR, FPS_JAHR, p19)

# 12h — Mittag (Sommer: Solar-Überschuss sichtbar)
p12 = os.path.join(EXP_CHARTS_DIR, f'EXP_kuer_k01d_{CC_CODE.lower()}_jahr_12h.gif')
make_gif_fast_d(lambda ax,fi,nf: _dyn_gif_b(ax,fi,nf,12), N_FRAMES_JAHR, FPS_JAHR, p12)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
from IPython.display import Image, display
import os
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01d_ch_jahr_19h.gif')
if os.path.exists(_p):
    display(Image(filename=_p))
else:
    print(f'⚠️  Datei nicht gefunden: {_p}')


**GIF B — Jahresverlauf Abendspitze (19h)**

Richtungsumkehr: CH-Saldo-Badge wechselt grün (Export) ↔ rot (Import).
19h: Abendspitze → Importrisiko im Winter. 12h: Mittag → Solar-Überschuss Sommer.

---
## 8. Multi-Country Erweiterung<a id='multi_K_01d'></a>

[↑ TOC](#toc_K_01d)

Anleitung und Code-Template für DE, AT, FR, IT. Alle Funktionen sind generisch — nur `CC_CODE` ändern.

### 8.1 Workflow für ein neues Land

1. **`COUNTRY_CONFIG` ergänzen** (in Initialisierungszelle):
```python
'DE': {
    'name': 'Deutschland',
    'osm_area': 'ISO3166-1"="DE',
    'bbox': (6.00, 47.20, 15.10, 55.10),
    'voltage_kv': [220, 380],
    'zone_mapping': {},     # ggf. Bundesland-Zonen definieren
    'zone_colors': {},
    'map_xlim': (6.0, 15.2),
    'map_ylim': (47.2, 55.2),
}
```

2. **`CC_CODE = 'DE'`** setzen → Notebook komplett neu ausführen

3. Optional **Baseline** für DE in `COUNTRY_BASELINES` hinterlegen
   (TSO-Umspannwerke: Amprion, TenneT, TransnetBW, 50Hertz)

4. **GIFs** erscheinen automatisch mit `kuer_k01d_de_*` Prefix

### 8.2 Datenqualität nach Land

| Land | Overpass-Qualität | Baseline vorhanden | Hinweis |
|------|------------------|--------------------|---------|
| CH | ★★★★★ | ✅ 27 Knoten | Swissgrid gut dokumentiert |
| DE | ★★★★☆ | ❌ (noch offen) | 4 TSOs, > 200 Knoten |
| AT | ★★★★☆ | ❌ | APG (~60 Knoten) |
| FR | ★★★☆☆ | ❌ | RTE, teilweise fehlend in OSM |
| IT | ★★★☆☆ | ❌ | Terna, OSM unvollständig |

### 8.3 PyPSA-Eur als beste Vollösung

Für DE/AT/FR/IT empfehlen sich die **prebuilt networks** von Zenodo:
- `zenodo.org/records/14144752` — alle CH/DE/AT/FR/IT buses + lines als CSV
- Direkt filterbar nach `country` Spalte
- Reaktanz/Widerstand vorhanden (für echten DC-Lastfluss)


In [ ]:
# ── Multi-Country Batch (alle konfigurierten Länder) ─────────────────────────
# Aktivieren mit: BATCH_RUN = True
# Erzeugt statische Karten für alle Länder mit verfügbaren Daten

BATCH_RUN = False  # ⚙ True für automatischen Multi-Country-Durchlauf

if BATCH_RUN:
    for cc, cc_cfg in COUNTRY_CONFIG.items():
        print(f"\n{'='*50}")
        print(f"=== {cc}: {cc_cfg['name']} ===")

        # Topologie laden
        result = download_grid_osm(cc, cc_cfg, EXP_CACHE_DIR, force=False)
        if result is None:
            result = load_from_baseline(cc, cc_cfg)
        if result is None:
            print(f"⚠️  Keine Daten für {cc} — übersprungen")
            continue

        df_n, df_e = result

        # Graph aufbauen
        G_cc = nx.Graph()
        for _, row in df_n.iterrows():
            G_cc.add_node(row['id'], lon=float(row['lon']), lat=float(row['lat']),
                          voltage_kv=int(row.get('voltage_kv',220)),
                          zone=str(row.get('zone','')))
        for _, row in df_e.iterrows():
            fn = row.get('from_node', row.get('id',''))
            tn = row.get('to_node', row.get('id',''))
            if fn in G_cc and tn in G_cc and fn != tn:
                G_cc.add_edge(fn, tn, voltage_kv=int(row.get('voltage_kv',220)))
        nx.set_node_attributes(G_cc, dict(G_cc.degree()), 'degree')

        # Statische Karte
        fig, ax = plt.subplots(figsize=(14, 9))
        fig.patch.set_facecolor(BG_DARK)
        ax.set_facecolor('#090d14')
        ax.set_xlim(*cc_cfg['map_xlim']); ax.set_ylim(*cc_cfg['map_ylim'])
        ax.set_axis_off()
        for n1, n2, data in G_cc.edges(data=True):
            kv  = data.get('voltage_kv', 220)
            x1,y1 = G_cc.nodes[n1]['lon'],G_cc.nodes[n1]['lat']
            x2,y2 = G_cc.nodes[n2]['lon'],G_cc.nodes[n2]['lat']
            ax.plot([x1,x2],[y1,y2], color=VOLTAGE_COLORS.get(kv,'#888'),
                    lw=VOLTAGE_LW.get(kv,1), alpha=0.7, zorder=2)
        for n, d in G_cc.nodes(data=True):
            size = max(10, min(250, d.get('degree',1)**2 * 10))
            ax.scatter(d['lon'], d['lat'], s=size, c=VOLTAGE_COLORS.get(d.get('voltage_kv',220),'#ccc'),
                       alpha=0.85, zorder=8, linewidths=0)
        ax.set_title(f'Netz {cc_cfg["name"]} ({cc}) | {G_cc.number_of_nodes()} Knoten',
                     color='white', fontsize=13, fontweight='bold')
        plt.tight_layout()
        p = os.path.join(EXP_CHARTS_DIR, f'EXP_kuer_k01d_{cc.lower()}_netz_statisch.png')
        plt.savefig(p, dpi=110, bbox_inches='tight', facecolor=BG_DARK)
        plt.close()
        print(f"✅ {p}")
else:
    print("BATCH_RUN = False — nur CH aktiv. Für alle Länder: BATCH_RUN = True setzen.")


---
## 9. Abschluss<a id='abschluss_K_01d'></a>

[↑ TOC](#toc_K_01d)

In [ ]:
# ── Abschlusskontrolle K_01d ─────────────────────────────────────────────────
print('K_01d – Abschlusskontrolle')
print('='*60)
cc = CC_CODE.lower()
expected = [
    f'EXP_kuer_k01d_{cc}_netz_statisch.png',
    f'EXP_kuer_k01d_{cc}_tag_winter.gif',
    f'EXP_kuer_k01d_{cc}_tag_frühling.gif',
    f'EXP_kuer_k01d_{cc}_tag_sommer.gif',
    f'EXP_kuer_k01d_{cc}_tag_herbst.gif',
    f'EXP_kuer_k01d_{cc}_jahr_19h.gif',
    f'EXP_kuer_k01d_{cc}_jahr_12h.gif',
]
total_kb = 0
ok = 0
for fname in expected:
    p = os.path.join(EXP_CHARTS_DIR, fname)
    exists = os.path.exists(p)
    kb = os.path.getsize(p)//1024 if exists else 0
    print(f"  {'✅' if exists else '❌'} {fname:<50} {kb:>5} KB")
    if exists: ok += 1; total_kb += kb

print()
print(f"  Total: {ok}/{len(expected)} | {total_kb//1024:.1f} MB")
print()
print(f"  Graph: {G.number_of_nodes()} Knoten, {G.number_of_edges()} Kanten")
print(f"  Quelle: {DATA_SOURCE}")
print()
print('  Multi-Country: BATCH_RUN = True → alle konfigurierten Länder')
print('  Neue Länder:  COUNTRY_CONFIG[\"XX\"] eintragen → CC_CODE = \"XX\"')


| [← K_01c Animationen](K_01c_Energiefluss_Animationen.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_02 Cross-Border →](K_02_Cross_Border.ipynb) |
|:---|:---:|---:|